# Model Persistence and Inference with Joblib
Lets now summarize how to train a Random Forest model on California housing data, save the model and preprocessing pipeline using joblib, and reuse the model later for inference on new data (input.csv). This approach helps avoid retraining the model every time, improving performance and enabling reproducibility.

### Why These Steps?
#### 1. Why Train Once and Save?
- Training models repeatedly is time-consuming and computationally expensive.
- Saving the model (model.pkl) and preprocessing pipeline (pipeline.pkl) ensures you can quickly load and run inference anytime in the future.
#### 2. Why Use a Preprocessing Pipeline?
- Raw data needs to be cleaned, scaled, and encoded before model training.
- A Pipeline automates this transformation and ensures identical preprocessing during inference.
#### 3. Why Use Joblib?
- joblib efficiently serializes large NumPy arrays (like in sklearn models).
- Faster and more suitable than pickle for scikit-learn objects.
#### 4. Why the If-Else Logic?
- The program checks if a saved model exists.


##### If not, it trains and saves the model.
##### If it does, it skips training and only runs inference, saving time.

##### Not run code for reference

In [ ]:
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor
from sklearn.compose import ColumnTransformer      
import pandas as pd
import numpy as np

import os
import joblib   # to save the model and pipeline


MODEL_PATH = "utils/model.pkl"
PIPELINE_PATH = "utils/pipeline.pkl"


def build_pipeline( nums_attribute ,catg_attributes):
    
    nums_pipeline = Pipeline([
        ('imputer' , SimpleImputer(strategy='median')),
        ('scaler' , StandardScaler())
    ])

    catg_pipeline = Pipeline([
        # ('encoder' , OriginalEncoder()),  -> if import it
        ('onehot_enc' , OneHotEncoder( handle_unknown='ignore'))   # handle_unknown -> if any new/catd data comes in test data then it (ignore it by making it 0 numerialy)
    ])


    full_pipeline = ColumnTransformer([
        ('nums' , nums_pipeline , nums_attribute),    # (name , pipeline , attribte_of that pipeline type)
        ('catg' , catg_pipeline , catg_attributes)
    ])


    return full_pipeline



if not os.path.exists(MODEL_PATH) :
    
    # Letts build and train model
    
    housing = pd.read_csv('california_housing_data.csv')
 
    housing[ 'income_categories'] = pd.cut( housing['median_income'] ,
                                        bins=[ 0 ,2 ,4, 6 , 8, np.inf],
                                            labels=[ 1 ,2 , 3, 4, 5 ] )


    shuffle_split = StratifiedShuffleSplit( n_splits=1 , test_size=0.2 , random_state=42)

    for test_split_indx , train_split_indx in shuffle_split.split( housing , housing[ 'income_categories'] ):
        train_data = housing.loc[train_split_indx]
        test_data = housing.loc[test_split_indx]
        

    train_data.drop( columns=['income_categories'] , inplace = True)   
    test_data.drop( columns=['income_categories'] , inplace = True)  
    

    # saving the testing data in cvsv
    test_data.to_csv("utils/testing_data.csv",index=False) 

    housing = train_data.copy()
    
    housing_label = housing['median_house_value'].copy()
    housing_features = housing.drop( columns=['median_house_value'] , axis=1)


    nums_attributes = housing_features.drop(columns=['ocean_proximity'] , axis=1).columns.to_list()
    catg_attribute = ['ocean_proximity']
    
    
    pipeline = build_pipeline(nums_attributes , catg_attribute)
    
    prepared_housing_numpy = pipeline.fit_transform(housing_features)
    
    housing_prepared = pd.DataFrame(prepared_housing_numpy ,
                                columns=['longitude', 'latitude', 'housing_median_age', 'total_rooms', 'total_bedrooms', 'population', 
                                         'households', 'median_income','INLAND','NEAR BAY','NEAR OCEAN','<1H OCEAN'] ,
                                index=housing_features.index)
    
    
    model = RandomForestRegressor(random_state=42)
    model.fit(housing_prepared , housing_label)
    
    #==================================================
    # Saving Model and pipeline as Pickle file -> used to persit the models/pipeline
    
    joblib.dump(model, MODEL_PATH)
    joblib.dump(pipeline , PIPELINE_PATH)
    #==================================================

    
    print("Model succesfully saved!")
    

else:
    # lets Inference ( test the model) 
    
    input_data = pd.read_csv("utils/testing_data.csv")
    
    #==================================================
    # Load the model and pipeline
    model = joblib.load(MODEL_PATH)
    pipeline = joblib.load(PIPELINE_PATH)
    #==================================================
    
    housing_test_features = input_data.drop( columns=['median_house_value'] , axis=1)

    transformed_input  = pipeline.transform(housing_test_features)     # preprocessing transform on test data as we do with train one
    
    prediction = model.predict(transformed_input)                 # predict "median_house_value"
    
    input_data['median_house_value'] = prediction
    
    input_data.to_csv('utils/predict_output.csv',index=False)
    print("Inference complete. Results saved to output.csv")


    

C:\Users\HP\AppData\Roaming\Python\Python313\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but RandomForestRegressor was fitted with feature names
  warnings.warn(


Inference complete. Results saved to output.csv


--------------------

### Summary

With this setup, our ML pipeline is:

1. Efficient – No retraining needed if the model exists.
2. Reproducible – Same preprocessing logic every time.
3. Production-ready – Can be deployed or reused across multiple systems.

--------------------

### Quick Easy Notes

- Joblib:
    in orderto save and load ML models.
    - Very popular with scikit-learn.
    - Better than pickle for large NumPy arrays and machine learning objects.

- Inference:
    - In ML, inference means using a trained model to make predictions on new data.
    - Example: after training, the model sees new input and gives an output/prediction.

- Pickle:
    - Pickle is Python’s built-in way to save/load objects.
    - It is okay for many simple projects.
    - For ML models, joblib is usually preferred because it is faster and more efficient.